# Eavesdropper Attacks on BB84

Generated from the public notebook builder for reproducible analysis.

Notebook 02 introduces Eve. We add a single function to `src/bb84.py` &mdash; `eve_intercept_resend` &mdash; and use it to verify two of the most quoted BB84 results:

1. **Full intercept-resend produces $\mathrm{QBER} = 25\%$** (not 50%, despite Eve guessing the wrong basis half the time).
2. **QBER is linear in Eve's interception probability:** $\mathrm{QBER}(p) = 0.25\,p$.

We then push to the security threshold and chart the key rate vs. interception probability for two error-correction efficiencies. All values are simulated end-to-end &mdash; nothing is hardcoded.

**Caveat:** every key-rate plot in this notebook uses the *idealized single-photon source model*. Real weak coherent sources are vulnerable to PNS attacks; that is discussed conceptually in Step 3.4 and addressed by decoy states.

## 1. Bootstrap and imports

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root()
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


Project root: C:\Users\COWLAR\projects\qkd-protocol-simulator


In [2]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from src.bb84 import (
    alice_prepare, bob_measure, sift, estimate_qber,
    error_correction, final_key_length, eve_intercept_resend,
)
from src.info_theory import binary_entropy

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

rng_master = np.random.default_rng(2026)
N = 100_000
